# Clasificación textos en Japonés
## 丁寧語 | 尊敬語 | 謙譲語





In [ ]:
!pip install fugashi[unidic-lite]

     ---------------------------------------- 0.0/47.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/47.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/47.4 MB ? eta -:--:--
     ---------------------------------------- 0.3/47.4 MB ? eta -:--:--
      --------------------------------------- 0.8/47.4 MB 1.6 MB/s eta 0:00:30
     - -------------------------------------- 1.3/47.4 MB 1.7 MB/s eta 0:00:27
     - -------------------------------------- 1.8/47.4 MB 1.9 MB/s eta 0:00:24
     -- ------------------------------------- 2.6/47.4 MB 2.4 MB/s eta 0:00:19
     --- ------------------------------------ 3.7/47.4 MB 2.7 MB/s eta 0:00:17
     --- ------------------------------------ 4.2/47.4 MB 2.7 MB/s eta 0:00:16
     ---- ----------------------------------- 5.0/47.4 MB 2.8 MB/s eta 0:00:15
     ---- ----------------------------------- 5.2/47.4 MB 2.7 MB/s eta 0:00:16
     ---- ----------------------------------- 5.5/47.4 MB 2.7 MB/s eta 0:00:16
 


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
# Convertir texto en números
from sklearn.feature_extraction.text import TfidfVectorizer
# Modelo, aprende patrones
from sklearn.naive_bayes import MultinomialNB
# Tokenización de palabras japonesas, separar palabras correctamente en una frase
from fugashi import Tagger

# Naive Bayes
#### Aprende patrones de palabras
#### Busca patrones estadísticos en palabras

In [22]:
# 'Tagger()' es el "Motor" que: lee japonés, detecta palabras, analiza gramática
# Crea "Etiquetas gramaticales"
# 食べる　-> Verbo; 本日 -> Sustantivo; は -> Partícula...
# Necesitamos crear primero la herramienta:
    # herramienta = Clase()
    # herramienta.hacer_algo()
# Conclusión: Creamos el analizador que separará y analizará las palabras japonesas

tagger = Tagger()

In [6]:
#Función que tokeniza un texto que le pasemos en 'text'

def tokenize_japanese(text):
    words = []

    # Fugashi analiza la frase y la tokeniza con Tagger()
    for word in tagger(text):
      # word.surface, fugashi guarda información tipo verbo, pronunciación, gramática, raíz, etc
        words.append(word.surface)

    return words

In [7]:
tokenize_japanese("私はあなたと会いました")

['私', 'は', 'あなた', 'と', '会い', 'まし', 'た']

In [8]:
try:
    df= pd.read_csv("/content/drive/MyDrive/日本語の例文271.csv")
    print("CSV file loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{"/content/drive/MyDrive/日本語の例文.csv"}' was not found. Please ensure it is exported as a CSV and saved in the correct path.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

CSV file loaded successfully!


,例文,分類
0,どうぞ、お掛けください,尊敬語
1,〇〇課長は、間もなくいらっしゃいます,尊敬語
2,社長のおっしゃることに共感いたしました,尊敬語
3,どのようなお仕事をなさるのですか,尊敬語
4,メールは、ご覧になりましたか,尊敬語


In [9]:
X = df["例文"]
y = df["分類"]

## TF-IDF
#### Da "pesos" a las palabras dependiendo de su importancia, de las veces que aparecen
###### Palabras importantes -> Más peso
###### Palabras comunes -> Menos peso
### Transforma texto en números inteligentes
### Convierte frases en listas numéricas, dependiendo de las veces que aparecen

In [25]:
#TF-IDF

# Herramienta que convierte texto en números
# 'tokenizer = tokenize_japanese' -> Usa mi función, no su separador de palabras normal
# Ya que sklearn está pensado para inglés (token_pattern=None para que no "se queje" por no usar su tokenizador)
vectorizer = TfidfVectorizer(tokenizer = tokenize_japanese, token_pattern = None)

# Aprendizaje de vocabulario
# 'X' son las frases -> X = df["例文"]
# Vectorizador analiza todo el data set y construye un diccionario interno
# Luego convierte frases en números
X_vectorized = vectorizer.fit_transform(X)

# X_vectorized es la versión numérica de las frases, matrices de números

In [11]:
# Crear Modelo (Creamos el "Cerebro vacío")
# El modelo aprenderá después relaciones entre datos
  # En este caso palabras japonesas con tipo de lenguaje
  # Algoritmo que aprende patrones de texto

# NB -> Naive Bayes
  # Fácil, rápido, bueno para NLP clásico
model = MultinomialNB()

# Aprendizaje Supervisado
### Le damos ejemplos con respuestas correctas
### Importantísimo tener un buen dataset con respuestas correctas

In [12]:
# Entrenar el modelo
# Aprende usando ejemplos
# X_vectorized -> Versión numérica de las frases
# y -> Las respuestas correctas
  # [0.2, 0.8...]	keigo **
  # [0.0, 0.1...]	casual **

# fit ajusta los datos, estudia los ejemplos *para aprender patrones*
# Calcula probabilidades *: P(keigo | contiene "ございます")　/ Qué tan probable es que sea keigo si aparece esa palabra
# Hace falta hacer primero TF-IDF, porque el modelo no entiende texto
model.fit(X_vectorized, y)

MultinomialNB()

# Predicción

In [13]:
# test_phrase = ["私が担当いたします"]
test_phrase = input("Escribe una frase en japonés: ")

test_vector = vectorizer.transform([test_phrase])

prediction = model.predict(test_vector)[0]

print("Tipo detectado: ", prediction)

Escribe una frase en japonés: 私が担当いたします
Tipo detectado:  謙譲語


## Prueba UI

In [14]:
from IPython.display import display
import ipywidgets as widgets

In [15]:

text_input = widgets.Text(
    placeholder="Escribe una frase en japonés",
    description="Frase:"
)

button = widgets.Button(description="Predecir")
output = widgets.Output()



In [16]:
def on_click(b):
    with output:
        output.clear_output()

        texto = text_input.value
        vec = vectorizer.transform([texto])
        pred = model.predict(vec)[0]

        print("Tipo detectado:", pred)

button.on_click(on_click)

In [17]:
display(text_input, button, output)

Text(value='', description='Frase:', placeholder='Escribe una frase en japonés')

Button(description='Predecir', style=ButtonStyle())

Output()

# TEST

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [19]:
# Dividir datos

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
# X_train

X_train_vectorized = vectorizer.fit_transform(X_train)

model.fit(X_train_vectorized, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


MultinomialNB()

In [21]:
# >70%-85%

X_test_vectorized = vectorizer.transform(X_test)

predictions = model.predict(X_test_vectorized)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.6481481481481481
